In [34]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [35]:
import torch
from torchmetrics import StructuralSimilarityIndexMeasure, MeanSquaredError
from torchmetrics.multimodal.clip_score import CLIPScore
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

from PIL import Image
from torch.nn import functional as F
from tqdm import tqdm
import json

In [63]:
img_dir = '相似的几个2'
res_dir = 'results/imgs_eval_no_control'
names = [name[:-len('.png')] for name in os.listdir(img_dir) if name.endswith('.png')]
result_dict = {name: {'path':os.path.join(img_dir, name+'.png'), 'result_list':[]} for name in names}
for res in os.listdir(res_dir):
    if 'control' in res:
        continue
    for name in result_dict.keys():
        if name in res:
            result_dict[name]['result_list'].append(os.path.join(res_dir, res))
            break
result_dict

{'Snipaste_2023-02-21_13-32-42': {'path': '相似的几个2/Snipaste_2023-02-21_13-32-42.png',
  'result_list': ['results/imgs_eval_no_control/Snipaste_2023-02-21_13-32-42_stage1_111.png',
   'results/imgs_eval_no_control/Snipaste_2023-02-21_13-32-42_stage1_222.png',
   'results/imgs_eval_no_control/Snipaste_2023-02-21_13-32-42_stage1_333.png',
   'results/imgs_eval_no_control/Snipaste_2023-02-21_13-32-42_stage1_444.png']},
 'Snipaste_2023-02-21_13-33-06': {'path': '相似的几个2/Snipaste_2023-02-21_13-33-06.png',
  'result_list': ['results/imgs_eval_no_control/Snipaste_2023-02-21_13-33-06_stage1_111.png',
   'results/imgs_eval_no_control/Snipaste_2023-02-21_13-33-06_stage1_222.png',
   'results/imgs_eval_no_control/Snipaste_2023-02-21_13-33-06_stage1_333.png',
   'results/imgs_eval_no_control/Snipaste_2023-02-21_13-33-06_stage1_444.png']},
 'Snipaste_2023-02-21_13-33-51': {'path': '相似的几个2/Snipaste_2023-02-21_13-33-51.png',
  'result_list': ['results/imgs_eval_no_control/Snipaste_2023-02-21_13-33-51_st

In [64]:
class ImagePairDataset(Dataset):
    def __init__(self, result_dict, normalize):
        self.image_path_pairs = []
        for v in result_dict.values():
            path = v['path']
            for res_path in v['result_list']:
                self.image_path_pairs.append((path, res_path))
        #self.result_dict = result_dict
        self.transform = transforms.ToTensor()
        self.normalize = normalize

    def __len__(self):
        return len(self.image_path_pairs)

    def __getitem__(self, idx):
        ori_img = Image.open(self.image_path_pairs[idx][0]).convert('RGB')
        res_img = Image.open(self.image_path_pairs[idx][1]).convert('RGB')
        res_img = res_img.resize(ori_img.size)
        ori_img = self.transform(ori_img)
        res_img = self.transform(res_img)
        if not self.normalize:
            ori_img = (ori_img * 255).to(torch.uint8)
            res_img = (res_img * 255).to(torch.uint8)
        return ori_img, res_img

In [65]:
def get_dataset_dataloader(batch_size, normalize = True):
    dataset = ImagePairDataset(result_dict, normalize=normalize)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return dataset, dataloader

In [66]:
batch_size = 4

In [67]:
ssim = StructuralSimilarityIndexMeasure(data_range=1.0)
normalize = True
dataset, dataloader = get_dataset_dataloader(batch_size=batch_size, normalize=True)
sum = 0
count = 0
for batch in tqdm(dataloader):
    sum += ssim(batch[0], batch[1])
    count += 1
ssim_res = sum.item() / count
ssim_res

100%|██████████| 5/5 [00:02<00:00,  1.76it/s]


0.23789045810699463

In [ ]:
mse = MeanSquaredError()
dataset, dataloader = get_dataset_dataloader(batch_size=batch_size, normalize=True)
sum = 0
count = 0
for batch in tqdm(dataloader):
    sum += mse(batch[0], batch[1])
    count += 1
mse_res = sum.item() / count
mse_res = 100*(1-mse_res)
mse_res

100%|██████████| 5/5 [00:01<00:00,  3.07it/s]


89.61234331130981

In [69]:
fid = FrechetInceptionDistance(feature=64, normalize=True)
dataset, dataloader = get_dataset_dataloader(batch_size=batch_size, normalize=True)
for batch in tqdm(dataloader):
    fid.update(batch[0], real=True)
    fid.update(batch[1], real=False)
fid_res = fid.compute().item()
fid_res

100%|██████████| 5/5 [00:01<00:00,  2.70it/s]


2.6437771320343018

In [ ]:
clip_score = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to('cuda')
dataset, dataloader = get_dataset_dataloader(batch_size=4, normalize=False)
sum = 0
count = 0
for batch in tqdm(dataloader):
    score = clip_score(batch[0], batch[1])
    sum += score
    count += 1
clip_res = sum.item() / count
clip_res

100%|██████████| 5/5 [00:11<00:00,  2.25s/it]


89.88172607421875

In [71]:
eval_dict = {'img_dir': img_dir, 'res_dir': res_dir, 'ssim': ssim_res, 'mse': mse_res, 'fid': fid_res, 'clip': clip_res}
with open(os.path.join(res_dir, 'result.json'), 'w', encoding='utf-8') as f:
    json.dump(eval_dict, f, ensure_ascii=False, indent=2)